# Notebook 03 — Enzyme Kinetics: Michaelis-Menten

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 03 of 10  
**Prerequisites:** NB02 (protein structure)  
**Time estimate:** 90 minutes

> **Pass-3 target:** The MM derivation and Lineweaver-Burk are both Pass-3 items.
> After working through this notebook, implement them from memory and record
> whether you succeeded unaided. This is the most common biochemistry interview question.

---
## Step 1 — Motivation

Almost every drug target is an enzyme. Drug development hinges on IC50 (inhibitor
concentration for 50% inhibition), which is derived from Km and Vmax. Understanding
these parameters is what separates someone who can read a pharmacology paper from
someone who can evaluate it critically.

---
## Step 2 — Intuition

Imagine a taxi rank with a fixed number of taxis (enzymes). When few passengers
(substrate molecules) arrive, most taxis wait idle — reaction rate is low and
proportional to passenger count. As more passengers arrive, taxis get busier.
Eventually all taxis are occupied at all times — the rank is saturated — and adding
more passengers doesn't increase throughput. Vmax is the throughput when all taxis
are busy. Km is the passenger count at which the rank is half-saturated.

---
## Step 3 — Biological Background

### Enzyme mechanism

$$E + S \underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}} ES \overset{k_{cat}}{\rightarrow} E + P$$

- E: free enzyme; S: substrate; ES: enzyme-substrate complex; P: product
- k₁: association rate; k₋₁: dissociation rate; k_cat: catalytic rate

### Inhibition types

| Inhibitor type | Binds to | Effect on Km | Effect on Vmax | Lineweaver-Burk |
|----------------|---------|-------------|---------------|------------------|
| Competitive | Free enzyme (same site as S) | ↑ (apparent) | Unchanged | Lines cross on y-axis |
| Non-competitive | ES complex (allosteric) | Unchanged | ↓ | Lines cross on x-axis |
| Uncompetitive | ES complex | ↓ (apparent) | ↓ | Parallel lines |
| Mixed | Both E and ES | ↑ or ↓ | ↓ | Lines cross left of y-axis |

### Hill equation — cooperative enzymes

Some enzymes (e.g. haemoglobin for O₂) show sigmoidal kinetics due to cooperativity:
$$v = \frac{V_{max}[S]^n}{K_d^n + [S]^n}$$
n > 1: positive cooperativity (sigmoidal); n = 1: Michaelis-Menten; n < 1: negative cooperativity.

---
## Step 4 — Mathematical Explanation

**Michaelis-Menten derivation (Briggs-Haldane steady-state):**

Assume d[ES]/dt = 0 (steady state):
$$k_1[E][S] = (k_{-1} + k_{cat})[ES]$$

Define Km = (k₋₁ + k_cat) / k₁. With [E]_total = [E] + [ES]:
$$[ES] = \frac{[E]_{total}[S]}{K_m + [S]}$$

Initial velocity v = k_cat · [ES]:
$$v = \frac{V_{max}[S]}{K_m + [S]}, \quad V_{max} = k_{cat}[E]_{total}$$

**Lineweaver-Burk (double-reciprocal):**
$$\frac{1}{v} = \frac{K_m}{V_{max}} \cdot \frac{1}{[S]} + \frac{1}{V_{max}}$$
- y-intercept: 1/Vmax
- x-intercept: −1/Km
- slope: Km/Vmax

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
# Cell 6.1 — Michaelis-Menten equation and variants
def michaelis_menten(S: np.ndarray, Vmax: float, Km: float) -> np.ndarray:
    """Standard Michaelis-Menten equation: v = Vmax*S/(Km+S)"""
    return Vmax * S / (Km + S)

def competitive_inhibition(S, Vmax, Km, I, Ki):
    """Competitive inhibitor: increases apparent Km by factor (1 + I/Ki)."""
    Km_app = Km * (1 + I / Ki)
    return Vmax * S / (Km_app + S)

def noncompetitive_inhibition(S, Vmax, Km, I, Ki):
    """Non-competitive inhibitor: decreases apparent Vmax, Km unchanged."""
    Vmax_app = Vmax / (1 + I / Ki)
    return Vmax_app * S / (Km + S)

def uncompetitive_inhibition(S, Vmax, Km, I, Ki):
    """Uncompetitive: both Km and Vmax decrease by same factor."""
    alpha = 1 + I / Ki
    return (Vmax / alpha) * S / ((Km / alpha) + S)

def hill_equation(S, Vmax, Kd, n):
    """Hill equation for cooperative binding. n>1 = positive cooperativity."""
    return Vmax * S**n / (Kd**n + S**n)

# Parameters
Vmax, Km = 10.0, 2.0
S_range = np.linspace(0.01, 20, 300)
v_noinhib = michaelis_menten(S_range, Vmax, Km)

print(f"MM parameters: Vmax={Vmax}, Km={Km}")
print(f"  At [S]=Km: v = {michaelis_menten(Km, Vmax, Km):.2f} = Vmax/2 ✓")
print(f"  At [S]=10*Km: v = {michaelis_menten(10*Km, Vmax, Km):.2f} ≈ Vmax = {Vmax} ✓")

In [ ]:
# Cell 6.2 — Parameter estimation from noisy data via Lineweaver-Burk and curve_fit
rng = np.random.default_rng(42)

# Simulate experimental data with noise
S_exp = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0])
v_true = michaelis_menten(S_exp, Vmax, Km)
v_noisy = v_true + rng.normal(0, 0.3, size=len(S_exp))
v_noisy = np.maximum(v_noisy, 0.01)  # no negative velocities

# Method 1: Lineweaver-Burk (double-reciprocal linear fit)
inv_S = 1 / S_exp
inv_v = 1 / v_noisy
slope_lb, intercept_lb = np.polyfit(inv_S, inv_v, 1)
Vmax_lb = 1 / intercept_lb
Km_lb = slope_lb * Vmax_lb

# Method 2: Non-linear least squares (preferred in practice)
popt, pcov = curve_fit(michaelis_menten, S_exp, v_noisy,
                        p0=[8, 1], bounds=([0, 0], [np.inf, np.inf]))
Vmax_nls, Km_nls = popt
perr = np.sqrt(np.diag(pcov))

print("Parameter estimation comparison:")
print(f"  True values:         Vmax={Vmax:.2f}, Km={Km:.2f}")
print(f"  Lineweaver-Burk:     Vmax={Vmax_lb:.2f}, Km={Km_lb:.2f}")
print(f"  Nonlinear LS:        Vmax={Vmax_nls:.2f}±{perr[0]:.2f}, Km={Km_nls:.2f}±{perr[1]:.2f}")
print("  Note: Lineweaver-Burk overweights low-[S] points — NLS is preferred in practice")

In [ ]:
# Cell 6.3 — Compare inhibition models
I = 5.0   # inhibitor concentration
Ki = 3.0  # inhibition constant

v_competitive    = competitive_inhibition(S_range, Vmax, Km, I, Ki)
v_noncompetitive = noncompetitive_inhibition(S_range, Vmax, Km, I, Ki)
v_uncompetitive  = uncompetitive_inhibition(S_range, Vmax, Km, I, Ki)

Km_app_comp = Km * (1 + I/Ki)
Vmax_app_noncomp = Vmax / (1 + I/Ki)
print(f"Inhibition at [I]={I}, Ki={Ki}:")
print(f"  Competitive:     Km_app = {Km_app_comp:.2f} (↑), Vmax unchanged = {Vmax}")
print(f"  Non-competitive: Km unchanged = {Km}, Vmax_app = {Vmax_app_noncomp:.2f} (↓)")
print(f"  Uncompetitive:   Km_app = {Km/(1+I/Ki):.2f} (↓), Vmax_app = {Vmax/(1+I/Ki):.2f} (↓)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — MM curve + Lineweaver-Burk + inhibition comparison + Hill cooperativity
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel A: MM curve with annotations
ax = axes[0, 0]
ax.plot(S_range, v_noinhib, color='steelblue', lw=2.5, label='MM curve')
ax.scatter(S_exp, v_noisy, color='tomato', zorder=5, s=40, label='Simulated data')
ax.plot(S_range, michaelis_menten(S_range, Vmax_nls, Km_nls), color='tomato',
        ls='--', lw=1.5, label='Fitted curve')
ax.axhline(Vmax, color='gray', ls=':', lw=1)
ax.axhline(Vmax/2, color='gray', ls=':', lw=1)
ax.axvline(Km, color='gray', ls=':', lw=1)
ax.annotate('', xy=(Km, Vmax/2), xytext=(0, Vmax/2),
            arrowprops=dict(arrowstyle='<->', color='seagreen', lw=1.5))
ax.text(Km/2, Vmax/2 + 0.3, f'← Km={Km} →', ha='center', color='seagreen', fontsize=8)
ax.text(S_range[-1]*0.5, Vmax*1.02, f'Vmax={Vmax}', fontsize=8, color='gray')
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title('A. Michaelis-Menten kinetics')
ax.legend(fontsize=8)

# Panel B: Lineweaver-Burk
ax = axes[0, 1]
S_lb_range = np.linspace(0.05, 2, 200)
v_lb = michaelis_menten(1/S_lb_range, Vmax, Km)

ax.scatter(1/S_exp, 1/v_noisy, color='tomato', s=40, zorder=5)
x_fit = np.linspace(-0.5, 1/S_exp.min() + 0.1, 200)
y_fit = slope_lb * x_fit + intercept_lb
ax.plot(x_fit, y_fit, color='steelblue', lw=2)
ax.axvline(0, color='black', lw=0.8); ax.axhline(0, color='black', lw=0.8)
ax.scatter([-1/Km_lb], [0], color='seagreen', s=60, zorder=6, label=f'-1/Km={-1/Km_lb:.2f}')
ax.scatter([0], [1/Vmax_lb], color='orange', s=60, zorder=6, label=f'1/Vmax={1/Vmax_lb:.2f}')
ax.set_xlabel('1/[S]'); ax.set_ylabel('1/v')
ax.set_title('B. Lineweaver-Burk plot')
ax.set_xlim(-0.55, 12); ax.set_ylim(-0.05, 0.7)
ax.legend(fontsize=8)

# Panel C: Three inhibition models
ax = axes[1, 0]
ax.plot(S_range, v_noinhib, color='steelblue', lw=2, label='No inhibitor')
ax.plot(S_range, v_competitive, color='tomato', lw=2, ls='--', label=f'Competitive (Km↑)')
ax.plot(S_range, v_noncompetitive, color='seagreen', lw=2, ls=':', label=f'Non-competitive (Vmax↓)')
ax.plot(S_range, v_uncompetitive, color='orange', lw=2, ls='-.', label=f'Uncompetitive (both↓)')
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title(f'C. Inhibition models ([I]={I}, Ki={Ki})')
ax.legend(fontsize=8)

# Panel D: Hill equation cooperativity
ax = axes[1, 1]
Kd = 2.0
for n, color in [(0.5, 'steelblue'), (1.0, 'gray'), (2.0, 'orange'), (4.0, 'tomato')]:
    v_hill = hill_equation(S_range, Vmax, Kd, n)
    style = '--' if n == 1.0 else '-'
    ax.plot(S_range, v_hill, color=color, lw=2, ls=style, label=f'n={n}')
ax.axhline(Vmax/2, color='gray', ls=':', lw=0.8)
ax.axvline(Kd, color='gray', ls=':', lw=0.8)
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title('D. Hill equation: cooperativity (Kd=2)')
ax.legend(fontsize=8)
ax.text(Kd + 0.2, Vmax/2 + 0.3, f'Kd={Kd}', fontsize=8, color='gray')

plt.suptitle('Enzyme Kinetics: Michaelis-Menten and Extensions', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `michaelis_menten(S, Vmax, Km)` from scratch. Without running code:
   what is the velocity at [S] = 0? At [S] = ∞? At [S] = Km?
2. Implement `competitive_inhibition()` and `noncompetitive_inhibition()`. Verify
   that competitive inhibition leaves the y-intercept of a Lineweaver-Burk plot
   unchanged (same 1/Vmax) but changes the slope.
3. A drug has IC50 = 50 nM and is a competitive inhibitor. Km (no inhibitor) = 10 µM.
   At [S] = Km and [I] = IC50, what fraction of the enzyme is inhibited?
4. The Hill coefficient for O₂ binding to haemoglobin is approximately n = 2.8.
   Why is n > 1 physiologically useful for oxygen delivery from lung to tissue?

---
## Quiz — Active Recall

1. State the Michaelis-Menten equation from memory.
2. What does Km represent physically? Does a lower Km mean higher or lower affinity?
3. In a Lineweaver-Burk plot, what do the x-intercept, y-intercept, and slope represent?
4. How does competitive inhibition change Km and Vmax?
5. What does a Hill coefficient of n = 4 imply about the binding mechanism?

---
## Papers Referenced

- Michaelis & Menten (1913); English translation: Goody & Johnson (2011)
- Briggs & Haldane (1925), Biochemical Journal

---
## Reflection — Pass 3 Record

**Date completed:** ____________________  
**Pass-3 attempt:** Close notebook. Write the MM equation and its derivation from scratch.  
**Succeeded unaided?** [ ] Yes  [ ] No — rebuilt from (describe what you needed to look up)

> *[One paragraph: explain MM kinetics to a software engineer who has never taken biochemistry.]*

---
*Next: `04_dna_replication_and_repair.ipynb`*